## Bike Sharing Demand

This project predicts the hourly demand for a bike-sharing system in Washington D.C. using historical rental counts, date/time, and weather data. It is a regression problem where the target (rental count) is always non-negative and often skewed, which is why the competition scores submissions on a logarithmic error.

## Approach
1. Load the data and explore rental patterns (by hour, day of week, season, weather)
2. Engineer time features from the datetime column (hour, day of week, month, year)
3. Train a baseline regression model and evaluate it with cross-validation, using the competition's metric (RMSLE)
4. Generate predictions and create the submission file
5. Submit to Kaggle and record the score

In [28]:
import os
from getpass import getpass

os.environ["KAGGLE_API_TOKEN"] = getpass("Paste your Kaggle API token (starts with KGAT_) and press Enter: ")

!pip install -q -U kaggle
!kaggle competitions download -c bike-sharing-demand
!unzip -oq bike-sharing-demand.zip

Paste your Kaggle API token (starts with KGAT_) and press Enter: ··········
100% 189k/189k [00:00<00:00, 78.7MB/s]



In [29]:
# Libraries
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import cross_val_score
from sklearn.metrics import make_scorer, mean_squared_log_error

In [31]:
# Load the data
train = pd.read_csv("train.csv", parse_dates=["datetime"])
test = pd.read_csv("test.csv", parse_dates=["datetime"])

In [32]:
train.shape, test.shape

((10886, 12), (6493, 9))

In [33]:
train.sample(1)

,datetime,season,holiday,workingday,weather,temp,atemp,humidity,windspeed,casual,registered,count
786,2011-02-16 05:00:00,1,0,1,1,8.2,9.85,47,12.998,0,5,5


In [34]:
train.isnull().sum()

,0
datetime,0
season,0
holiday,0
workingday,0
weather,0
temp,0
atemp,0
humidity,0
windspeed,0
casual,0


In [35]:
test.isnull().sum()

,0
datetime,0
season,0
holiday,0
workingday,0
weather,0
temp,0
atemp,0
humidity,0
windspeed,0


In [36]:
train["hour"], train["dow"], train["month"], train["year"] = train["datetime"].dt.hour, train["datetime"].dt.dayofweek, train["datetime"].dt.month, train["datetime"].dt.year

In [37]:
test["hour"], test["dow"], test["month"], test["year"] = test["datetime"].dt.hour, test["datetime"].dt.dayofweek, test["datetime"].dt.month, test["datetime"].dt.year

In [38]:
train.sample(1)

,datetime,season,holiday,workingday,weather,temp,atemp,humidity,windspeed,casual,registered,count,hour,dow,month,year
6967,2012-04-08 15:00:00,2,0,0,1,25.42,30.305,17,23.9994,260,226,486,15,6,4,2012


In [39]:
test.sample(1)

,datetime,season,holiday,workingday,weather,temp,atemp,humidity,windspeed,hour,dow,month,year
2333,2011-09-28 18:00:00,4,0,1,3,27.06,30.305,83,16.9979,18,2,9,2011


In [40]:
# Defining x and y
x = train[["season", "holiday", "workingday", "weather", "temp", "atemp", "humidity", "windspeed", "hour", "dow", "month", "year"]]
y = np.log1p(train["count"])

In [41]:
x.shape, y.shape

((10886, 12), (10886,))

In [42]:
x.head()

,season,holiday,workingday,weather,temp,atemp,humidity,windspeed,hour,dow,month,year
0,1,0,0,1,9.84,14.395,81,0.0,0,5,1,2011
1,1,0,0,1,9.02,13.635,80,0.0,1,5,1,2011
2,1,0,0,1,9.02,13.635,80,0.0,2,5,1,2011
3,1,0,0,1,9.84,14.395,75,0.0,3,5,1,2011
4,1,0,0,1,9.84,14.395,75,0.0,4,5,1,2011


In [43]:
from sklearn.model_selection import train_test_split

cols = ["season", "holiday", "workingday", "weather", "temp", "atemp", "humidity", "windspeed", "hour", "dow", "month", "year"]

tr, val = train_test_split(train, test_size=0.2, random_state=42)

model = RandomForestRegressor(n_estimators=300, max_depth=12, random_state=42)
model.fit(tr[cols], np.log1p(tr["count"]))

pred = np.clip(np.expm1(model.predict(val[cols])), 0, None)
score = np.sqrt(mean_squared_log_error(val["count"], pred))
print("Validation RMSLE:", round(score, 4))

Validation RMSLE: 0.3092


In [44]:
model = RandomForestRegressor(n_estimators=300, max_depth=12, random_state=42)
model.fit(train[cols], np.log1p(train["count"]))

pred = np.clip(np.expm1(model.predict(test[cols])), 0, None)

pd.DataFrame({"datetime": test["datetime"], "count": pred}).to_csv("submission.csv", index=False)

In [45]:
!kaggle competitions submit -c bike-sharing-demand -f submission.csv -m "RandomForest baseline"

100% 244k/244k [00:00<00:00, 1.15MB/s]
99 submissions remaining today.
Successfully submitted to Bike Sharing Demand

In [46]:
import pickle
pickle.dump(model, open("bike_model.pkl", "wb"))